<a href="https://colab.research.google.com/github/ulyssesMonte/Machine-Learning---House-Prices---Advanced-Regression-Techniques/blob/main/IFES_ML_House_Prices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desafio Kaggle: House Prices - Advanced Regression Techniques
Aluno: [Ulysses Monte]

Objetivo: Prever o preço de venda de residências em Ames, Iowa, utilizando técnicas avançadas de regressão e aprendizado de máquina.

### 1. Introdução
O mercado imobiliário é influenciado por diversos fatores, como características estruturais, localização e qualidade do imóvel. O conjunto de dados utilizado possui 79 variáveis explicativas para prever o preço de venda dos imóveis.

O principal desafio consiste no pré-processamento dos dados, incluindo o tratamento de valores ausentes, a codificação de variáveis categóricas e a construção de um modelo capaz de minimizar o RMSLE (Root Mean Squared Logarithmic Error), métrica utilizada pelo Kaggle para avaliar as previsões.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
# Modelo
from sklearn.ensemble import RandomForestRegressor

pd.set_option('display.max_columns', None)

### 2. Exploração Inicial dos Dados

Nesta etapa, foram carregados os conjuntos de dados de treino e teste para analisar sua estrutura. Em seguida, verificaram-se as dimensões, os tipos de dados e a variável alvo do problema, **SalePrice** (preço de venda dos imóveis).


In [5]:
dados_treino = pd.read_csv('train.csv')
dados_teste = pd.read_csv('test.csv')

# Verificar a proporção treino/teste
print(f"Dimensões do Treino: {dados_treino.shape}")
print(f"Dimensões do Teste: {dados_teste.shape}")

dados_treino.head()

Dimensões do Treino: (1460, 81)
Dimensões do Teste: (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [7]:
dados_treino.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

### 3. Preparação dos Dados

#### 3.1 Transformação da Variável Alvo

A variável **SalePrice** apresenta uma distribuição assimétrica, com poucos imóveis de alto valor. Para reduzir essa assimetria e melhorar o desempenho do modelo, foi aplicada a transformação logarítmica `log(1 + x)` utilizando `np.log1p`. Após a realização das previsões, os valores foram convertidos para a escala original com `np.expm1`.


In [8]:
alvo_log = np.log1p(dados_treino['SalePrice'])
caracteristicas_treino = dados_treino.drop(['SalePrice'], axis=1)

dados_completos = pd.concat([caracteristicas_treino, dados_teste], axis=0)
print(f"Dimensões da base unificada para tratamento: {dados_completos.shape}")

Dimensões da base unificada para tratamento: (2919, 80)


#### 3.2 Tratamento de Dados Ausentes

Nas variáveis numéricas, os valores nulos foram substituídos pela mediana da coluna. Variáveis categóricas tiveram os valores ausentes foram substituídos por **"None"**.


In [9]:
colunas_numericas = dados_completos.select_dtypes(include=['int64', 'float64']).columns

for col in colunas_numericas:
    dados_completos[col] = dados_completos[col].fillna(dados_completos[col].median())

colunas_categoricas = dados_completos.select_dtypes(include=['object']).columns

for col in colunas_categoricas:
    dados_completos[col] = dados_completos[col].fillna('None')

dados_processados = pd.get_dummies(dados_completos)
print(f"Novas dimensões após o One-Hot Encoding: {dados_processados.shape}")

Novas dimensões após o One-Hot Encoding: (2919, 311)


#### 3.3 Separação dos Conjuntos de Dados

Após o pré-processamento, dados separados novamente em conjuntos de treino e teste, utilizando o número de registros do conjunto de treino original como referência.


In [10]:
X_treino = dados_processados.iloc[:len(dados_treino)]

X_teste = dados_processados.iloc[len(dados_treino):]

print(f"Matriz de Recursos de Treino (X_treino): {X_treino.shape}")
print(f"Matriz de Recursos de Teste (X_teste): {X_teste.shape}")

Matriz de Recursos de Treino (X_treino): (1460, 311)
Matriz de Recursos de Teste (X_teste): (1459, 311)


### 4. Aplicação das Técnicas de Aprendizado de Máquina

#### 4.1 Escolha do Modelo

Utilizado o **Random Forest Regressor**, que apresenta bom desempenho em problemas de regressão e lida bem com grandes quantidades de variáveis. Também é menos suscetível a overfitting por combinar diversas árvores de decisão para gerar uma única previsão.

#### 4.2 Validação do Modelo

O desempenho foi avaliado por validação cruzada com 5 divisões (**5-fold cross-validation**). O conjunto de treino é dividido em cinco partes, permitindo que o modelo seja treinado e validado em diferentes subconjuntos. Essa técnica fornece uma estimativa da capacidade de generalização do modelo.


In [11]:
# Hiperparâmetros definidos para controle de complexidade
modelo_rf = RandomForestRegressor(
    n_estimators=500,  # 500 árvores
    max_depth=20,      # Limite de profundidade
    random_state=42,   # Semente
    n_jobs=-1          # Processadores da máquina em paralelo
)

# Scores
pontuacoes_rmse = -cross_val_score(
    modelo_rf,
    X_treino,
    alvo_log,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

print(f"RMSE de cada dobra: {pontuacoes_rmse}")
print(f"👉 Média do RMSE (Validação Cruzada): {pontuacoes_rmse.mean():.5f}")

RMSE de cada dobra: [0.13751183 0.15465542 0.14480845 0.12840221 0.14955646]
👉 Média do RMSE (Validação Cruzada): 0.14299


### 5. Avaliação dos Resultados e Geração da Submissão

Algoritmo treinado utilizando todo o conjunto de treino. E então realizadas as previsões para o conjunto de teste e gerado o arquivo de submissão.


In [12]:
modelo_rf.fit(X_treino, alvo_log)

predicoes_log = modelo_rf.predict(X_teste)

predicoes_finais = np.expm1(predicoes_log)

submissao = pd.DataFrame({
    'Id': dados_teste['Id'],
    'SalePrice': predicoes_finais
})

submissao.to_csv('submission.csv', index=False)

submissao.head()

,Id,SalePrice
0,1461,124805.895801
1,1462,153436.513567
2,1463,177371.912207
3,1464,181047.438207
4,1465,196471.101803


## 6. Conclusão

### 6.1 Resultados Obtidos

O modelo **Random Forest Regressor** apresentou um bom desempenho na validação cruzada, alcançando um RMSE médio satisfatório. O tratamento dos valores ausentes e a conversão das variáveis categóricas contribuíram para o treinamento do modelo, enquanto a transformação logarítmica da variável **SalePrice** ajudou a melhorar a qualidade das previsões.

### 6.2 Trabalhos Futuros

Como melhorias para o projeto, podem ser realizadas novas etapas de engenharia de atributos, seleção das variáveis mais relevantes e testes com modelos mais avançados. Também é possível otimizar os hiperparâmetros do modelo para buscar um desempenho superior.
